# 🐱🐶 Classification d'Images : Chats vs Chiens (avec Data Augmentation)

## Présentation du projet

Dans ce notebook, nous allons construire un **CNN de classification binaire** capable de distinguer les chats des chiens, et étudier l'impact de la **data augmentation** sur les performances.

### 🎯 Ce que vous allez apprendre :
- Prétraiter des images pour un CNN (générateurs Keras)
- Appliquer la **data augmentation** pour améliorer la généralisation
- Construire et entraîner un CNN de classification binaire
- Utiliser le **Dropout** pour réduire l'overfitting

### 🗺️ Feuille de route
1. Chargement des données + générateurs Keras
2. Inspection de l'équilibre des classes et des échantillons
3. Conception du CNN
4. Configuration de l'optimisation
5. Entraînement (avec et sans early stopping)
6. Évaluation sur la validation
7. Inférence sur le jeu de test non labellisé
8. Comparaison Baseline vs Augmentation
9. Gestion du déséquilibre de classes
10. Sauvegarde du modèle
11. Pistes d'extension

---
> ⚠️ **Important** : Ce notebook nécessite une VM (ex: DigitalOcean) ou Google Colab avec GPU. Le dataset doit être téléchargé manuellement, dézippé, renommé en `cats_dogs`, et placé dans un dossier `data/`.

---
## 📦 PARTIE 1 — Chargement des données et générateurs

**Section pré-remplie** — à copier et exécuter directement. Ce bloc :
- Découvre les fichiers sous `cats_dogs/train/train` et `cats_dogs/test/test`
- Déduit les labels depuis le nom du dossier parent ou le nom du fichier (`cat.123.jpg`)
- Construit 3 générateurs : `train_flow` (avec augmentation), `val_flow` (rescale seul), `test_flow` (non labellisé)

### 💡 Pourquoi c'est important
On standardise la taille des images et on met à l'échelle les pixels (0-1) pour stabiliser les gradients. L'augmentation expose le modèle à des transformations plausibles et réduit l'overfitting. **On n'augmente JAMAIS la validation ou le test** — seulement l'entraînement.

In [ ]:
# Pré-rempli. Copier et exécuter.
import os, math, re, random
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

np.random.seed(42); tf.random.set_seed(42)

# Chemins - à modifier si besoin
DATA_ROOT = Path("data/cats_dogs")
train_dir = (DATA_ROOT / "train" / "train") if (DATA_ROOT / "train" / "train").exists() else (DATA_ROOT / "train")
test_dir  = (DATA_ROOT / "test"  / "test")  if (DATA_ROOT / "test"  / "test").exists()  else (DATA_ROOT / "test")

# ── Résolution réduite ──────────────────────────────────────────────────────
# Le dataset original a une résolution très élevée → on la réduit pour
# accélérer l'entraînement (48×48 est suffisant pour ce projet pédagogique)
IMG_HEIGHT, IMG_WIDTH = 48, 48
batch_size = 32
seed = 1337

# Construire les DataFrames depuis les dossiers
def build_df_from_folder(folder: Path, labeled: bool = True):
    exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))
    if not files:
        raise FileNotFoundError(f"Aucune image trouvée sous {folder}")
    rows = []
    for f in files:
        if labeled:
            name = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {"cat", "cats"}:
                label = "cat"
            elif parent in {"dog", "dogs"}:
                label = "dog"
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name): label = "cat"
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name): label = "dog"
                else:
                    continue
            rows.append({"filepath": f, "label": label})
        else:
            rows.append({"filepath": f})
    return pd.DataFrame(rows)

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full  = build_df_from_folder(test_dir,  labeled=False)

# Split train / validation
from sklearn.model_selection import train_test_split
df_tr, df_val = train_test_split(
    df_train_full, test_size=0.2, stratify=df_train_full["label"], random_state=seed
)

# Générateurs
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True,
)
val_gen  = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    df_tr, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=True, seed=seed, validate_filenames=False
)
val_flow = val_gen.flow_from_dataframe(
    df_val, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=False, validate_filenames=False
)
# Test non labellisé, pour inférence uniquement
test_flow = test_gen.flow_from_dataframe(
    df_test_full, x_col="filepath", y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None, batch_size=batch_size,
    shuffle=False, validate_filenames=False
)

print({"train": train_flow.samples, "val": val_flow.samples, "test": test_flow.samples,
       "class_indices": train_flow.class_indices})

---
## 🔍 PARTIE 2 — Inspection des données

**To-Do** : analyser l'équilibre des classes, comprendre les sources de variabilité visuelle, et visualiser des exemples annotés.

### Pourquoi c'est important
Il faut connaître l'équilibre des classes pour décider d'une stratégie de pondération. Comprendre la variabilité visuelle (pose, échelle, éclairage, fond) guide les choix d'augmentation.

In [ ]:
# ── Analyse de l'équilibre des classes ───────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style='darkgrid')
%matplotlib inline

print(f"🔑 Mapping des classes : {train_flow.class_indices}")

# train_flow.labels contient le label (0 ou 1) pour chaque image d'entraînement
train_label_counts = Counter(train_flow.labels)
class_names_inv = {v: k for k, v in train_flow.class_indices.items()}

print("\n📊 Distribution des classes (entraînement) :")
for class_idx, count in sorted(train_label_counts.items()):
    pct = 100 * count / len(train_flow.labels)
    print(f"   {class_names_inv[class_idx]:<6} : {count:>5} images ({pct:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
names_plot = [class_names_inv[k] for k in sorted(train_label_counts)]
counts_plot = [train_label_counts[k] for k in sorted(train_label_counts)]
bars = ax.bar(names_plot, counts_plot, color=['#e67e22', '#3498db'], edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, counts_plot):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(v), ha='center', fontweight='bold')
ax.set_title("Distribution des classes — Chats vs Chiens", fontsize=12)
ax.set_ylabel("Nombre d'images")
plt.tight_layout()
plt.show()

### 📝 Analyse de l'équilibre des classes

*(à compléter avec vos résultats réels)*

**Nombre d'images par classe** : d'après `train_flow.class_indices` et `train_flow.labels`, le dataset Cats vs Dogs original de Kaggle est habituellement **parfaitement équilibré** (environ 50% chats, 50% chiens), avec un total de ~10 000 images après le split train/validation (80/20).

**Équilibre ou déséquilibre** : si la distribution observée ci-dessus montre un ratio proche de 50/50, les classes sont **équilibrées** — aucune pondération de classe n'est nécessaire. Si un déséquilibre apparaît (ex: à cause de fichiers corrompus filtrés différemment par classe), il faudra utiliser `class_weight` (voir Partie 9).

**Sources de variabilité visuelle potentielles** :
- **Pose** : animaux de face, de profil, couchés, en mouvement
- **Échelle** : photos en gros plan vs animal lointain dans la scène
- **Éclairage** : lumière naturelle, flash, intérieur sombre
- **Arrière-plan** : intérieur, extérieur, présence d'humains ou d'objets
- **Race et couleur** : grande variété de robes/pelages au sein de chaque classe

Cette variabilité justifie une **data augmentation agressive** (rotation, zoom, flip) pour que le modèle apprenne des features robustes plutôt que de mémoriser des poses spécifiques.

In [ ]:
# ── Grille d'images annotées ──────────────────────────────────────────────────
# On désactive temporairement le shuffle pour avoir un batch reproductible
sample_batch_x, sample_batch_y = next(train_flow)

fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for i, ax in enumerate(axes.flatten()):
    if i >= len(sample_batch_x):
        ax.axis('off')
        continue
    img   = sample_batch_x[i]
    label = int(sample_batch_y[i])
    label_name = class_names_inv[label]
    ax.imshow(img)
    ax.set_title(label_name, fontsize=11, fontweight='bold',
                color='#e67e22' if label_name == 'cat' else '#3498db')
    ax.axis('off')

plt.suptitle('Échantillons annotés (avec augmentation appliquée)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Notez les déformations dues à l'augmentation : rotation, zoom, décalage, flip horizontal.")
print("   Ces variations aident le modèle à généraliser au-delà des images exactes vues à l'entraînement.")

### 📝 Indices visuels pour distinguer chats et chiens

En observant la grille ci-dessus, plusieurs indices visuels peuvent aider le modèle :
- **Forme des oreilles** : triangulaires et pointues (chat) vs tombantes ou variées (chien)
- **Forme du museau** : court et plat (chat) vs allongé (chien, selon la race)
- **Texture de la fourrure** : souvent plus fine et lisse chez le chat
- **Posture corporelle** : silhouette plus élancée et souple chez le chat
- **Contexte** : les chiens apparaissent plus souvent en extérieur (laisse, jeu), les chats en intérieur

Un CNN apprend ces indices automatiquement via ses filtres convolutifs, sans qu'on ait besoin de les coder explicitement — c'est tout l'intérêt du deep learning par rapport aux features manuelles.

---
## 🧠 PARTIE 3 — Définition de l'architecture du modèle

### 📝 Description de l'architecture (avant codage)

**Blocs convolutifs** : on utilise **4 blocs convolutifs** avec un nombre croissant de filtres (32 → 64 → 128 → 128), ce qui suit la pratique standard où les couches profondes détectent des features de plus en plus abstraites et nécessitent davantage de filtres pour les capturer.

**Placement du MaxPooling** : chaque bloc convolutif est suivi d'une couche `MaxPooling2D(2,2)`, qui réduit les dimensions spatiales de moitié à chaque fois. Avec une entrée de 48×48, on obtient successivement 24×24 → 12×12 → 6×6 → 3×3. Le pooling apporte de l'**invariance à la translation** (un chat décalé de quelques pixels reste reconnu) et réduit drastiquement le nombre de paramètres.

**Dropout** : on insère du **Dropout(0.5)** après l'aplatissement (Flatten), juste avant les couches denses. Le Dropout désactive aléatoirement des neurones pendant l'entraînement, empêchant le réseau de devenir trop dépendant de combinaisons spécifiques de neurones — c'est une régularisation efficace contre l'overfitting, particulièrement utile ici car le dataset est de taille modeste comparé à la capacité du réseau.

**Couches denses finales** : après le Flatten et le Dropout, une couche `Dense(512, activation='relu')` agrège l'information spatiale, suivie de la couche de sortie `Dense(1, activation='sigmoid')` — un seul neurone avec activation sigmoid, car c'est une classification **binaire** (probabilité d'appartenir à la classe "dog", par exemple).

> 💡 **Point clé** : La *binary cross-entropy* est la loss appropriée pour modéliser une cible de Bernoulli avec une sortie sigmoid. Le Softmax est réservé aux problèmes multi-classes (3 classes ou plus).

In [ ]:
# ── Construction du CNN ───────────────────────────────────────────────────────
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

def build_cnn(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), dropout_rate=0.5):
    """
    CNN pour classification binaire (chat vs chien).

    Architecture :
        4 blocs [Conv2D → MaxPooling2D] avec filtres croissants (32→64→128→128)
        → Flatten → Dropout → Dense(512) → Dense(1, sigmoid)
    """
    model = Sequential([
        # Bloc 1
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        MaxPooling2D(2, 2),

        # Bloc 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(2, 2),

        # Bloc 3
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(2, 2),

        # Bloc 4
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D(2, 2),

        # Tête de classification
        Flatten(),
        Dropout(dropout_rate),     # Régularisation contre l'overfitting
        Dense(512, activation='relu'),
        Dense(1, activation='sigmoid')  # Sortie binaire : probabilité d'être un "dog"
    ])
    return model

model = build_cnn()
model.summary()

---
## ⚙️ PARTIE 4 — Configuration de l'optimisation

### 📝 Choix et justifications

**Optimiseur** : **Adam** est choisi pour sa convergence rapide et stable. Il adapte automatiquement le learning rate par paramètre en combinant les avantages de Momentum et RMSProp — un bon choix par défaut pour la plupart des tâches de vision.

**Learning rate initial** : `1e-3` (0.001), la valeur par défaut d'Adam, qui constitue un bon compromis entre vitesse de convergence et stabilité pour un CNN de cette taille. On utilisera `ReduceLROnPlateau` pour l'ajuster automatiquement si la validation stagne.

**Batch size** : `32`, un choix standard qui équilibre la stabilité du gradient (plus grand = plus stable) et la consommation mémoire (plus petit = moins de RAM/VRAM requise). Sur CPU ou GPU modeste, 32 est un bon point de départ.

**EarlyStopping** : surveille la `val_loss` avec `patience=10` — arrête l'entraînement si la validation loss ne s'améliore plus pendant 10 epochs, évitant ainsi l'overfitting et économisant du temps de calcul. `ReduceLROnPlateau` réduit le learning rate de moitié si la val_loss stagne pendant 5 epochs, permettant d'affiner la convergence en fin d'entraînement.

> 💡 **Point clé** : surveillez à la fois la **loss** et l'**accuracy**. L'accuracy peut être trompeuse en cas de déséquilibre de classes ; la loss est plus lisse et plus sensible à la qualité des probabilités prédites.

In [ ]:
# ── Compilation du modèle ─────────────────────────────────────────────────────
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',   # Loss correcte pour sortie sigmoid + cible binaire
    metrics=['accuracy']
)

# ── Callbacks ──────────────────────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_model.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=0
)

print("✅ Modèle compilé avec Adam (lr=1e-3) + binary_crossentropy")
print("✅ Callbacks configurés : EarlyStopping, ReduceLROnPlateau, ModelCheckpoint")

---
## 🚀 PARTIE 5 — Entraînement du modèle

**To-Do** : entraîner d'abord pour un nombre fixe d'epochs (sans early stopping), observer les courbes, puis ré-entraîner avec early stopping activé.

In [ ]:
# ── Entraînement 1 : nombre fixe d'epochs (sans early stopping) ─────────────
EPOCHS_FIXED = 30

model_fixed = build_cnn()
model_fixed.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])

print(f"🏋️  Entraînement sur {EPOCHS_FIXED} epochs fixes (sans early stopping)...")
history_fixed = model_fixed.fit(
    train_flow,
    validation_data=val_flow,
    epochs=EPOCHS_FIXED,
    verbose=1
)

In [ ]:
# ── Visualisation : entraînement à epochs fixes ──────────────────────────────
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(history.history['loss'],     label='Train',      color='tomato',    linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Validation', color='royalblue', linewidth=2, linestyle='--')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

    axes[1].plot(history.history['accuracy'],     label='Train',      color='tomato',    linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Validation', color='royalblue', linewidth=2, linestyle='--')
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

plot_history(history_fixed, "Entraînement à epochs fixes (sans Early Stopping)")

In [ ]:
# ── Entraînement 2 : avec Early Stopping ──────────────────────────────────────
EPOCHS_MAX = 50  # Maximum, mais EarlyStopping arrêtera avant si besoin

model = build_cnn()  # Nouveau modèle, poids réinitialisés
model.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])

print(f"🏋️  Entraînement avec Early Stopping (max {EPOCHS_MAX} epochs)...")
history = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=EPOCHS_MAX,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

print(f"\n✅ Entraînement arrêté après {len(history.history['loss'])} epochs.")

In [ ]:
plot_history(history, "Entraînement avec Early Stopping")

### 📝 Détection de l'overfitting et mitigation

**Comment détecter l'overfitting sur les courbes** : le signal classique est une **divergence croissante** entre la courbe de train et celle de validation — la loss d'entraînement continue de baisser tandis que la loss de validation stagne, puis remonte. De même pour l'accuracy : un écart qui se creuse entre train_accuracy (qui monte vers 95-99%) et val_accuracy (qui plafonne plus bas) signale que le modèle mémorise les exemples d'entraînement plutôt que d'apprendre des patterns généralisables.

**Changements pour mitiger l'overfitting** si on l'observe :
1. **Renforcer l'augmentation** : augmenter `rotation_range`, `zoom_range`, ajouter `shear_range` ou `brightness_range`
2. **Augmenter le Dropout** : passer de 0.5 à 0.6-0.7 dans la couche dense
3. **Réduire la capacité du modèle** : diminuer le nombre de filtres ou de neurones dans la couche Dense(512)
4. **Ajouter de la régularisation L2** (`kernel_regularizer`) sur les couches denses
5. **Early Stopping** (déjà en place) : restaure les poids du meilleur epoch avant que l'overfitting ne s'installe

L'entraînement **avec Early Stopping** devrait montrer un écart train/val plus contenu, car l'entraînement s'arrête dès que la validation loss cesse de s'améliorer, évitant la phase de sur-mémorisation.

---
## 📊 PARTIE 6 — Évaluation sur les données de validation

**To-Do** : calculer accuracy/loss de validation, matrice de confusion, précision/rappel par classe.

In [ ]:
# ── Évaluation globale ────────────────────────────────────────────────────────
val_loss, val_accuracy = model.evaluate(val_flow, verbose=0)
print(f"📊 Validation Loss     : {val_loss:.4f}")
print(f"📊 Validation Accuracy : {val_accuracy*100:.2f}%")

In [ ]:
# ── Matrice de confusion et rapport de classification ────────────────────────
from sklearn.metrics import confusion_matrix, classification_report

val_flow.reset()  # S'assurer que le générateur recommence depuis le début
val_probs = model.predict(val_flow, verbose=0).flatten()
val_preds = (val_probs > 0.5).astype(int)
val_true  = val_flow.labels

cm = confusion_matrix(val_true, val_preds)
class_labels = [class_names_inv[0], class_names_inv[1]]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=ax)
ax.set_title('Matrice de Confusion — Validation', fontsize=12)
ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
plt.tight_layout(); plt.show()

print("\n📋 Rapport de classification :")
print(classification_report(val_true, val_preds, target_names=class_labels, digits=3))

### 📝 Analyse du type d'erreur dominant

*(à compléter avec vos résultats réels — exemple de structure d'analyse)*

En observant la matrice de confusion, on regarde si les **faux positifs** (chiens classés chats) ou les **faux négatifs** (chats classés chiens) dominent. Si, par exemple, le modèle confond davantage les chats avec les chiens que l'inverse, cela peut indiquer un **biais de texture/couleur** appris par le modèle — certains chats à poil court ou de couleur similaire aux chiens présents dans le dataset sont mal discriminés.

**Implications pour le choix du seuil** : si le coût d'un faux négatif est plus élevé que celui d'un faux positif (ou inversement) dans votre cas d'usage, il est pertinent d'ajuster le seuil de décision (0.5 par défaut) plutôt que de le considérer comme fixe. Par exemple, abaisser le seuil à 0.4 favoriserait le rappel de la classe "dog" au prix d'une précision réduite.

**Implications pour l'augmentation** : si l'erreur dominante concerne des poses ou angles spécifiques, cela suggère d'enrichir l'augmentation avec des transformations supplémentaires (cisaillement, variations de luminosité) ciblant ces cas difficiles.

> 💡 **Point clé** : pour une sortie sigmoid, le seuil de 0.5 est **arbitraire**. Il faut le calibrer ou l'ajuster pour optimiser la métrique qui correspond réellement à l'exigence du problème (précision, rappel, F1, ou coût métier spécifique).

---
## 🔮 PARTIE 7 — Inférence sur le jeu de test non labellisé

**To-Do** : générer des probabilités sur `test_flow`, les convertir en labels avec un seuil justifié, et exporter un CSV.

In [ ]:
# ── Inférence sur le test non labellisé ──────────────────────────────────────
test_flow.reset()
test_probs = model.predict(test_flow, verbose=1).flatten()

# ── Seuil de décision ─────────────────────────────────────────────────────────
# On garde 0.5 par défaut (pas de coût asymétrique connu a priori),
# mais ce choix doit être justifié selon le contexte d'usage.
THRESHOLD = 0.5
test_preds = (test_probs > THRESHOLD).astype(int)

# class_indices : {'cat': 0, 'dog': 1} généralement
dog_idx = train_flow.class_indices.get('dog', 1)
pred_labels = [class_names_inv[p] for p in test_preds]

results_df = pd.DataFrame({
    'filepath':   df_test_full['filepath'].values,
    'prob_dog':   test_probs if dog_idx == 1 else (1 - test_probs),
    'pred_label': pred_labels
})

results_df.to_csv('test_predictions.csv', index=False)
print(f"✅ Prédictions sauvegardées dans 'test_predictions.csv'")
print(f"\n📋 Aperçu :")
results_df.head(10)

In [ ]:
# ── Vérification manuelle d'un sous-ensemble (sanity check) ──────────────────
sample_indices = np.random.choice(len(results_df), size=8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.flatten(), sample_indices):
    row = results_df.iloc[idx]
    img = plt.imread(row['filepath'])
    ax.imshow(img)
    ax.set_title(f"{row['pred_label']} (p_dog={row['prob_dog']:.2f})", fontsize=10)
    ax.axis('off')

plt.suptitle('Vérification manuelle — échantillon aléatoire de prédictions', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("💡 Méthode de vérification : on inspecte visuellement un échantillon aléatoire")
print("   et on compare la prédiction au jugement humain. Toute incohérence flagrante")
print("   (ex: chien clairement visible classé 'cat' avec haute confiance) signale un")
print("   problème potentiel de données ou de modèle à investiguer plus en profondeur.")

---
## ⚖️ PARTIE 8 — Comparaison Baseline vs Augmentation

**To-Do** : entraîner un modèle baseline (même architecture, sans augmentation) et comparer.

In [ ]:
# ── Générateur d'entraînement SANS augmentation (baseline) ───────────────────
train_gen_baseline = ImageDataGenerator(rescale=1./255)  # Rescale seul, pas d'augmentation

train_flow_baseline = train_gen_baseline.flow_from_dataframe(
    df_tr, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=True, seed=seed, validate_filenames=False
)

print("✅ Générateur baseline créé (rescale seul, sans augmentation).")

In [ ]:
# ── Entraînement du modèle baseline (architecture identique) ─────────────────
model_baseline = build_cnn()
model_baseline.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])

early_stop_baseline = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr_baseline  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

print("🏋️  Entraînement du modèle BASELINE (sans augmentation)...")
history_baseline = model_baseline.fit(
    train_flow_baseline,
    validation_data=val_flow,
    epochs=EPOCHS_MAX,
    callbacks=[early_stop_baseline, reduce_lr_baseline],
    verbose=1
)

print(f"\n✅ Baseline arrêtée après {len(history_baseline.history['loss'])} epochs.")

In [ ]:
# ── Comparaison visuelle Baseline vs Augmentation ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_baseline.history['loss'],     label='Train (baseline)',      color='tomato',  linewidth=2)
axes[0].plot(history_baseline.history['val_loss'], label='Val (baseline)',        color='tomato',  linewidth=2, linestyle='--')
axes[0].plot(history.history['loss'],              label='Train (augmenté)',      color='royalblue', linewidth=2)
axes[0].plot(history.history['val_loss'],          label='Val (augmenté)',        color='royalblue', linewidth=2, linestyle='--')
axes[0].set_title('Loss — Baseline vs Augmentation'); axes[0].set_xlabel('Epoch'); axes[0].legend(fontsize=8)

axes[1].plot(history_baseline.history['accuracy'],     label='Train (baseline)',  color='tomato',  linewidth=2)
axes[1].plot(history_baseline.history['val_accuracy'], label='Val (baseline)',    color='tomato',  linewidth=2, linestyle='--')
axes[1].plot(history.history['accuracy'],              label='Train (augmenté)',  color='royalblue', linewidth=2)
axes[1].plot(history.history['val_accuracy'],          label='Val (augmenté)',    color='royalblue', linewidth=2, linestyle='--')
axes[1].set_title('Accuracy — Baseline vs Augmentation'); axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

baseline_val_loss, baseline_val_acc = model_baseline.evaluate(val_flow, verbose=0)
print(f"\n📊 Comparaison finale :")
print(f"   Baseline  → Val Loss: {baseline_val_loss:.4f} | Val Acc: {baseline_val_acc*100:.2f}%")
print(f"   Augmenté  → Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy*100:.2f}%")

### 📝 Analyse du generalization gap

*(structure d'analyse — à compléter avec vos résultats réels)*

Le **generalization gap** se mesure par l'écart entre les performances d'entraînement et de validation. On s'attend typiquement à observer :

- **Modèle baseline (sans augmentation)** : un écart train/val plus large, car le modèle voit toujours les mêmes images exactes à chaque epoch et peut mémoriser des détails spécifiques (texture de fond, angle précis) qui ne généralisent pas.
- **Modèle augmenté** : un écart train/val plus resserré, car chaque epoch présente des versions transformées différentes des mêmes images, empêchant la mémorisation pure et forçant l'apprentissage de features plus robustes.

Si la val_accuracy du modèle augmenté est égale ou supérieure à celle du baseline **malgré une train_accuracy plus faible**, c'est le signe classique que l'augmentation a réussi : elle a rendu la tâche d'entraînement plus difficile artificiellement, mais a amélioré la capacité de généralisation — exactement l'effet recherché.

> 💡 **Point clé** : préférez les changements qui améliorent la performance de validation sans augmentation massive du temps de calcul. L'augmentation de données en est un exemple typique : elle améliore souvent la robustesse sans ajouter de paramètres au modèle.

---
## ⚖️ PARTIE 9 — Gestion du déséquilibre de classes

**To-Do** : si les classes sont déséquilibrées, calculer les poids de classe et ré-entraîner.

In [ ]:
# ── Calcul des poids de classe ────────────────────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

classes_array = np.unique(train_flow.labels)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=classes_array,
    y=train_flow.labels
)
class_weight_dict = dict(zip(classes_array, class_weights_array))

print("⚖️  Poids de classe calculés :")
for idx, weight in class_weight_dict.items():
    print(f"   {class_names_inv[idx]:<6} (classe {idx}) : poids = {weight:.3f}")

if abs(class_weight_dict[0] - class_weight_dict[1]) < 0.05:
    print("\n💡 Les poids sont proches de 1.0 → les classes sont quasi équilibrées.")
    print("   La pondération aura un effet minime dans ce cas précis.")
else:
    print("\n💡 Un déséquilibre notable est détecté → la pondération devrait aider")
    print("   à améliorer le rappel de la classe minoritaire.")

In [ ]:
# ── Ré-entraînement avec class_weight ─────────────────────────────────────────
model_weighted = build_cnn()
model_weighted.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])

early_stop_w = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("🏋️  Entraînement avec class_weight...")
history_weighted = model_weighted.fit(
    train_flow,
    validation_data=val_flow,
    epochs=EPOCHS_MAX,
    class_weight=class_weight_dict,
    callbacks=[early_stop_w],
    verbose=1
)

In [ ]:
# ── Comparaison precision/recall avant et après pondération ─────────────────
val_flow.reset()
val_probs_weighted = model_weighted.predict(val_flow, verbose=0).flatten()
val_preds_weighted = (val_probs_weighted > 0.5).astype(int)

print("📋 SANS pondération de classe :")
print(classification_report(val_true, val_preds, target_names=class_labels, digits=3))

print("\n📋 AVEC pondération de classe :")
print(classification_report(val_true, val_preds_weighted, target_names=class_labels, digits=3))

### 📝 Effet de la pondération sur précision et rappel

Si les classes sont déjà équilibrées (cas attendu pour Cats vs Dogs Kaggle), l'effet de `class_weight` sera **minime** : les poids calculés seront proches de 1.0 pour les deux classes, et les métriques resteront similaires.

Dans un scénario où une classe est réellement minoritaire, on observerait typiquement :
- **Hausse du rappel** de la classe minoritaire (moins de faux négatifs, car le modèle est pénalisé plus fortement pour ses erreurs sur cette classe)
- **Légère baisse de la précision** de la classe minoritaire (le modèle devient plus "généreux" en prédisant cette classe, augmentant les faux positifs)
- **Effet inverse, plus faible**, sur la classe majoritaire

Ce compromis precision/recall est exactement l'effet recherché par la pondération : elle empêche le réseau de simplement favoriser la classe majoritaire pour minimiser la loss globale.

---
## 💾 PARTIE 10 — Sauvegarde des artefacts

**To-Do** : sauvegarder le meilleur modèle et la configuration d'entraînement.

In [ ]:
# ── Sauvegarde du modèle et de la configuration ──────────────────────────────
import json

# 1. Sauvegarde du modèle (format Keras natif, recommandé)
model.save('cats_dogs_final_model.keras')
print("💾 Modèle sauvegardé → cats_dogs_final_model.keras")

# 2. Sauvegarde de la configuration d'entraînement (métadonnées)
training_config = {
    'img_height':     IMG_HEIGHT,
    'img_width':      IMG_WIDTH,
    'batch_size':     batch_size,
    'optimizer':      'Adam',
    'learning_rate':  1e-3,
    'loss':           'binary_crossentropy',
    'epochs_run':     len(history.history['loss']),
    'class_indices':  train_flow.class_indices,
    'val_loss':       float(val_loss),
    'val_accuracy':   float(val_accuracy),
    'augmentation': {
        'rotation_range':     45,
        'width_shift_range':  0.15,
        'height_shift_range': 0.15,
        'zoom_range':         0.5,
        'horizontal_flip':    True
    },
    'decision_threshold': THRESHOLD
}

with open('training_config.json', 'w') as f:
    json.dump(training_config, f, indent=2)

print("💾 Configuration sauvegardée → training_config.json")
print("\n📋 Contenu de la configuration :")
print(json.dumps(training_config, indent=2))

### 📝 Pourquoi sauvegarder à la fois les poids et les métadonnées

Sauvegarder uniquement les **poids du modèle** ne suffit pas pour garantir la **reproductibilité** : sans connaître la résolution d'image attendue (48×48), le type de normalisation appliqué, les paramètres d'augmentation utilisés, ou le seuil de décision choisi, il est impossible de réutiliser correctement le modèle plus tard ou de comprendre dans quelles conditions il a été évalué.

Les **métadonnées** (fichier JSON) permettent de :
- **Reproduire exactement le pipeline de prétraitement** lors du déploiement
- **Auditer** les choix faits (seuil, augmentation) en cas de question ultérieure
- **Comparer objectivement** plusieurs expériences (avec/sans augmentation, avec/sans pondération)
- **Faciliter le transfert** du modèle à un autre membre de l'équipe sans connaissance préalable du code source

---
## 🚀 PARTIE 11 — Pistes d'extension

**To-Do** : proposer une extension et justifier le bénéfice attendu.

Nous implémentons ici le **Transfer Learning avec MobileNetV2**, l'extension la plus impactante pour ce projet.

In [ ]:
# ── Extension : Transfer Learning avec MobileNetV2 ───────────────────────────
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model

# MobileNetV2 attend généralement des images plus grandes (96×96 minimum)
# On agrandit légèrement la résolution pour cette extension
TRANSFER_SIZE = 96

base_model = MobileNetV2(
    input_shape=(TRANSFER_SIZE, TRANSFER_SIZE, 3),
    include_top=False,       # On retire la tête de classification ImageNet (1000 classes)
    weights='imagenet'       # Poids pré-entraînés sur ImageNet
)
base_model.trainable = False  # On gèle le backbone (feature extraction)

transfer_model = Model(inputs=base_model.input, outputs=Dense(1, activation='sigmoid')(
    Dense(128, activation='relu')(
        GlobalAveragePooling2D()(base_model.output)
    )
))

transfer_model.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])
transfer_model.summary()

print(f"\n✅ MobileNetV2 chargé (backbone gelé)")
print(f"   Paramètres entraînables : {sum(np.prod(w.shape) for w in transfer_model.trainable_weights):,}")
print(f"   Paramètres totaux       : {transfer_model.count_params():,}")

### 📝 Justification de l'extension choisie

**Transfer Learning avec MobileNetV2 gelé** apporte plusieurs bénéfices attendus :

1. **Features de bas niveau réutilisables** : MobileNetV2 a été pré-entraîné sur ImageNet (1,2M images, 1000 classes), apprenant des détecteurs de bords, textures et formes qui sont **universels** pour la vision par ordinateur — ces features sont directement pertinentes pour distinguer chats et chiens, sans avoir besoin de les réapprendre depuis zéro.

2. **Convergence plus rapide** : avec un backbone gelé, seules les couches de la tête de classification (quelques milliers de paramètres) sont entraînées, ce qui réduit drastiquement le temps de calcul et le risque d'overfitting sur un dataset de taille modeste.

3. **Meilleure précision attendue** : malgré sa légèreté (MobileNetV2 est conçu pour le mobile), ce backbone capture des représentations bien plus riches qu'un CNN entraîné from scratch sur seulement quelques milliers d'images.

> 💡 **Point clé** : le transfer learning fournit de forts a priori pour les features de bas niveau (bords, textures) communes à toutes les images naturelles. Le fine-tuning des couches supérieures permet ensuite d'adapter le modèle à la tâche spécifique.

---
## ✅ PARTIE 12 — Checklist des livrables

| Livrable | Statut | Section |
|----------|--------|---------|
| Rapport de données (comptage classes + grille d'exemples) | ✅ | Partie 2 |
| Description du modèle et justification de l'optimisation | ✅ | Parties 3-4 |
| Courbes d'entraînement et interprétation | ✅ | Parties 5, 8 |
| Métriques de validation (matrice de confusion, precision/recall) | ✅ | Partie 6 |
| CSV des prédictions test (probabilités + labels) | ✅ | Partie 7 (`test_predictions.csv`) |
| Modèle sauvegardé + log d'exécution | ✅ | Partie 10 (`cats_dogs_final_model.keras` + `training_config.json`) |

---
## 🎓 Conclusion

Vous avez construit un **pipeline complet de classification binaire** sur des images réelles :

| Partie | Ce qu'on a fait |
|--------|----------------|
| **1** | Générateurs Keras avec augmentation (train) et sans (val/test) |
| **2** | Analyse de l'équilibre des classes et visualisation d'exemples annotés |
| **3** | CNN à 4 blocs convolutifs + Dropout + sortie sigmoid |
| **4** | Adam, EarlyStopping, ReduceLROnPlateau justifiés |
| **5** | Entraînement à epochs fixes vs avec Early Stopping |
| **6** | Matrice de confusion et rapport precision/recall |
| **7** | Export CSV des prédictions test avec sanity check visuel |
| **8** | Comparaison quantitative Baseline vs Augmentation |
| **9** | Calcul et application de `class_weight` |
| **10** | Sauvegarde modèle (.keras) + métadonnées (.json) |
| **11** | Transfer Learning avec MobileNetV2 |

### 💡 Rappels clés
- N'augmentez **jamais** la validation ou le test, seulement l'entraînement
- Gardez un split de validation fixe pour comparer équitablement vos expériences
- En cas de résultats suspects, **vérifiez d'abord les labels et chemins** — les bugs de données dominent largement les bugs de modèle